# 第 2 堂　第一支程式：讓電腦自己排序

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rthtsai/junior-lab/blob/main/w02.ipynb)

**怎麼用**：由上往下，一格一格按左邊的 ▶ 執行。

- 看到 **✏️** 的格子，要自己把 `____` 換成答案。沒填就按，會出現紅色錯誤，這是正常的。
- 卡住的順序：先看格子裡的提示 → 問同組 → 再舉手。
- 上週你們用 AI 排序。今天換成**自己寫程式排序**：檢索（排序）→ 依據（印出是哪一段）→ 評分（Recall@2 由程式算）。

## 先做這一格（第 1 節一開始就按）

下載語意模型要 1 到 3 分鐘，先讓它在背景跑，後面第 2 節才會用到。
如果跳出「要重新啟動工作階段」，按「重新啟動」再繼續，不用重按這格。

In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")
print("語意模型準備好了")

---
## 第 1 節　字面比對檢索器

### 步驟 1　把資料放進程式

`question` 是題目，`passages` 是五段短文。這是上週的咖啡題。

In [ ]:
question = "為什麼下午喝的咖啡，到了晚上還會影響睡眠？"
passages = {
    "F": "咖啡因在人體內的半衰期約 5 到 6 小時。下午三點喝下的一杯咖啡，到了晚上九點，體內仍有大約一半的咖啡因在作用。",
    "G": "咖啡因的作用方式是佔住大腦中接收「睏意訊號」的受體，讓這個訊號傳不進去，於是人感覺不到累。",
    "H": "咖啡豆的風味會因產地而有明顯差異。淺焙通常保留較多酸味，深焙則偏苦甜。",
    "I": "有些人天生代謝咖啡因的速度較慢，同樣的分量對他們的影響會持續更久。",
    "J": "睡前喝酒雖然比較容易入睡，但會讓後半夜的睡眠變淺、容易醒來。"
}

print(question)
print("一共", len(passages), "段")

### ✏️ 1　用迴圈看每一段

`for ... in ...` 會把每一段依序拿出來。請填空，讓它印出每一段**有幾個字**。

提示：`len(某個字串)` 會回傳字數。

In [ ]:
for name, text in passages.items():
    print(name, "共", len(____), "個字")

### 步驟 2　寫一個「字面比對」函式

想法：題目裡的字，有幾個也出現在這一段？越多，分數越高。這就是上週說的「字面重疊」。

In [ ]:
STOP = set("的了嗎呢啊是在有為什麼要會到就都也還和與及或、，。？！：；「」（）() 0123456789")

def overlap(question, passage):
    q = set(question) - STOP    # 題目裡出現過的字（不重複，去掉常見虛字）
    p = set(passage) - STOP     # 這一段裡出現過的字
    # ✏️ 2　回傳「兩邊都有的字」有幾個。提示：q & p 是兩邊都有的字
    return len(____)

### 步驟 3　幫每一段打分數、排名

In [ ]:
scores = {}
for name, text in passages.items():
    scores[name] = overlap(question, text)

ranking = sorted(scores, key=scores.get, reverse=True)   # 分數高的排前面
print("分數：", scores)
print("排名：", " > ".join(ranking))

### ✏️ 3　用程式算 Recall@2

上週你們用手算。現在寫成函式：前 k 名裡，命中幾個正解，除以正解總數。

提示：`hits` 已經算好了，只要補完最後一行。

In [ ]:
gold = ["F", "G"]      # 這一題的正解

def recall_at_k(ranking, gold, k):
    top = ranking[:k]                      # 前 k 名
    hits = len(set(top) & set(gold))       # 前 k 名裡命中幾個正解
    return ____ / len(gold)

print("字面比對　Recall@2 =", recall_at_k(ranking, gold, 2))

### 想一想（回表單用）

1. 排在前面、但其實答不到題目的是哪一段？它為什麼拿到高分？
2. 真正的正解（F、G）排第幾？程式為什麼沒把它們排前面？

---
## 第 2 節　語意比對檢索器

字面比對只看「字一不一樣」。語意比對換一種做法：先把每一段話交給模型，轉成**一串數字**（向量）；意思越近的兩句話，數字越像。

In [ ]:
vec = model.encode(["咖啡因會讓人睡不著"], normalize_embeddings=True)
print("一句話變成", vec.shape[1], "個數字")
print("前 5 個：", vec[0][:5])

下面這個函式已經寫好：把題目和每一段都變成數字，再算「有多像」。你只要會用它。

In [ ]:
def semantic_scores(question, passages):
    names = list(passages)
    q_vec = model.encode([question], normalize_embeddings=True)
    p_vec = model.encode([passages[n] for n in names], normalize_embeddings=True)
    sims = (p_vec @ q_vec.T).ravel()      # 兩個向量越像，數字越大
    return dict(zip(names, [float(s) for s in sims]))

### ✏️ 4　用語意比對排咖啡題

把題目放進去，其他照著前面的寫法。

In [ ]:
sem_scores = semantic_scores(____, passages)
sem_ranking = sorted(sem_scores, key=sem_scores.get, reverse=True)
print("分數：", {k: round(v, 3) for k, v in sem_scores.items()})
print("排名：", " > ".join(sem_ranking))
print("語意比對　Recall@2 =", recall_at_k(sem_ranking, gold, 2))

---
## 應用任務　兩種檢索器，三個題組

每個題組有一個題目、六段（或五段）短文，正解在 `gold`。下面這格會讓兩種方法各跑一遍。

In [ ]:
SETS = {
    "題組 1　咖啡（上週的題目）": {
        "q": "為什麼下午喝的咖啡，到了晚上還會影響睡眠？",
        "gold": [
            "F",
            "G"
        ],
        "p": {
            "F": "咖啡因在人體內的半衰期約 5 到 6 小時。下午三點喝下的一杯咖啡，到了晚上九點，體內仍有大約一半的咖啡因在作用。",
            "G": "咖啡因的作用方式是佔住大腦中接收「睏意訊號」的受體，讓這個訊號傳不進去，於是人感覺不到累。",
            "H": "咖啡豆的風味會因產地而有明顯差異。淺焙通常保留較多酸味，深焙則偏苦甜。",
            "I": "有些人天生代謝咖啡因的速度較慢，同樣的分量對他們的影響會持續更久。",
            "J": "睡前喝酒雖然比較容易入睡，但會讓後半夜的睡眠變淺、容易醒來。"
        }
    },
    "題組 2　雷雨": {
        "q": "為什麼夏天的雷雨常常在下午發生？",
        "gold": [
            "B",
            "D"
        ],
        "p": {
            "A": "夏天下午的雷雨來得又急又猛，常常伴隨強風，戶外活動時要記得隨身帶傘，並且避開空曠處與大樹下。",
            "B": "中午過後，地面被太陽曬得很熱，貼近地面的空氣受熱上升，水氣在高空冷卻凝結，容易發展成積雨雲。",
            "C": "夏天白晝比冬天長，台北夏至的日出約在清晨五點多，日落則要到傍晚近七點。",
            "D": "一天之中，氣溫通常在午後兩、三點最高，此時低層大氣最不穩定，上升氣流最強，積雨雲也最容易發展。",
            "E": "下午茶的習慣起源於十九世紀的英國，用餐時間延後，貴族們在午餐與晚餐之間補充點心與茶。",
            "F": "雷聲是閃電瞬間加熱周圍空氣，使空氣急速膨脹所發出的聲音。光比聲音快，所以人會先看到閃電，過幾秒才聽到雷聲。"
        }
    },
    "題組 3　紙箱": {
        "q": "為什麼貓咪喜歡鑽進紙箱？",
        "gold": [
            "A",
            "C"
        ],
        "p": {
            "A": "貓在狹小、被包住的空間裡會覺得有安全感，研究也發現，有紙箱可躲的貓，壓力指標明顯比沒有的貓低。",
            "B": "貓咪鑽紙箱的影片在網路上很受歡迎，許多飼主會特地買紙箱放在家裡，讓貓咪隨時都有紙箱可以鑽。",
            "C": "貓覺得舒服的環境溫度約在攝氏三十度以上，比人類高。厚紙板隔熱效果不錯，躲進裡面比較容易保暖。",
            "D": "貓的眼睛構造讓牠在昏暗環境中的視力比人類好很多，所以黃昏和清晨特別活躍。",
            "E": "貓用臉頰蹭人或家具，是在留下費洛蒙氣味，標記這裡是自己熟悉而安心的地方。",
            "F": "貓一天大約睡十二到十六小時，睡眠時間比人類長很多，幼貓還會睡得更久。"
        }
    }
}

for title, d in SETS.items():
    kw  = {n: overlap(d["q"], t) for n, t in d["p"].items()}
    sem = semantic_scores(d["q"], d["p"])
    r_kw  = sorted(kw,  key=kw.get,  reverse=True)
    r_sem = sorted(sem, key=sem.get, reverse=True)
    print(title)
    print("  題目：", d["q"])
    print("  正解：", d["gold"])
    print("  字面比對：", " > ".join(r_kw),  "  Recall@2 =", recall_at_k(r_kw,  d["gold"], 2))
    print("  語意比對：", " > ".join(r_sem), "  Recall@2 =", recall_at_k(r_sem, d["gold"], 2))
    print()

### 要交的（填在表單）

1. 三個題組，兩種方法的 Recall@2 各是多少？
2. 挑一個題組：哪一段「排在前面卻答不到」？它騙到了字面比對、語意比對，還是兩個都騙到？
3. 用一句話說：為什麼它會騙到？（提示：想想上週的 A 段）

### 想改進的話（選做）

試試看改下面的東西，重跑上一格，記下 Recall@2 有沒有變好：

- 在 `STOP` 裡多加幾個字
- 把題目改寫得更具體
- 把 `"BAAI/bge-m3"` 換成別的模型

---
## 下週要交：自己出一題

在下面照格式寫一個題組：題目一個、六段短文，其中**至少一段要「字面很像題目，但其實答不到」**，再指定兩段正解。寫完用兩種方法各跑一次，記下 Recall@2，看看你的干擾段騙得到誰。

In [ ]:
my_set = {
    "q": "在這裡寫你的題目",
    "gold": ["A", "B"],          # 兩段正解的編號
    "p": {
        "A": "正解 1",
        "B": "正解 2",
        "C": "干擾段：字面很像題目，但答不到",
        "D": "干擾段 2",
        "E": "干擾段 3",
        "F": "干擾段 4",
    },
}

kw  = {n: overlap(my_set["q"], t) for n, t in my_set["p"].items()}
sem = semantic_scores(my_set["q"], my_set["p"])
r_kw  = sorted(kw,  key=kw.get,  reverse=True)
r_sem = sorted(sem, key=sem.get, reverse=True)
print("字面比對：", " > ".join(r_kw),  "Recall@2 =", recall_at_k(r_kw,  my_set["gold"], 2))
print("語意比對：", " > ".join(r_sem), "Recall@2 =", recall_at_k(r_sem, my_set["gold"], 2))